# EDS full-sequence stride comparison
Run all cells. Auto mode reuses each completed score and checks input file signatures. Plot-only reads the previous report for this selection. Stride 1 uses the original full-frame predictions. Every method and stride is scored on the same half-open interval for a sequence.

Fast version: scoring runs in `score_strides_fast.py` (the same scoring code, with sequence/stride pairs in parallel and shared GT KD-trees). Cache keys are unchanged, so earlier results are reused.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
from functools import cached_property
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "experiments/score_windows.py").is_file())
sys.path.insert(0, str(ROOT / "experiments"))
import score_strides_fast
from score_windows import Window, bench

from voxel_metric_v2 import PROTOCOL, score as voxel_score, time_edges
import voxel_metric_v2

VOXEL_METRICS = ("voxel_mae", "voxel_rmse", "voxel_relative_l1", "pmse_2_v2ce_adapted", "pmse_4_v2ce_adapted")
UNITS = {**bench.UNITS, **{m: "squared events/voxel" for m in VOXEL_METRICS if m.startswith("pmse")}}
VOXEL_BIN_MS = 10.0  # Nominal bin width; the last bin can be shorter.
# Append either adapted PMSE name above to evaluate it (dense grids can require substantial RAM).
METRICS = ["chamfer_full", "event_count_ratio", "event_rate", "iei_distance", "fano_factor",
           "voxel_mae", "voxel_rmse", "voxel_relative_l1"]
METHODS = ["v2e", "iebcs", "dvs_voltmeter", "v2v", "evis_core", "fracevent", "evtween_v5.2_student"]
SEQUENCES = ["02_rocket_earth_light", "06_ziggy_and_fuzz", "10_office", "14_ziggy_in_the_arena"]
EXPERIMENTS = {1: None, 2: "stride_m1", 3: "stride_m2", 4: "stride_m3"}
MODE = "auto"  # "auto" or "plot_only"
DATA = bench.v2.old.setting("EVTWEEN_DATA", bench.v2.old.LAB / "evtween/data")
DEVICE, WORKERS, VOXEL_BINS = "cpu", 16, 160  # DEVICE "cuda": Chamfer on CuPy (same definition, last digits may differ)
PROCESSES = 8  # Parallel (sequence, stride) pairs; each holds a full sequence and its GT trees in RAM
REPORT = ROOT / "reports/experiments/eds/test/full_sequence_strides"

## Common evaluation interval and scoring
The endpoint is the earliest final selected timestamp among all four conditions. Count-based frame bins use the original timeline within this shared interval. Native V2V signed voxel planes cannot recover event timestamps or polarity-separated counts and are marked unavailable for these event-stream protocols. ISI distribution is retained as an array; its scalar view is the per-sequence median ISI in ms.

In [ ]:
# Scoring code: experiments/score_strides_fast.py (options, measure and input_dir as in the original notebook).
def run_scores():
    return score_strides_fast.run_scores(
        METRICS=METRICS, METHODS=METHODS, SEQUENCES=SEQUENCES, EXPERIMENTS=EXPERIMENTS, DATA=DATA,
        DEVICE=DEVICE, WORKERS=WORKERS, VOXEL_BINS=VOXEL_BINS, VOXEL_BIN_MS=VOXEL_BIN_MS, REPORT=REPORT,
        processes=PROCESSES)


In [ ]:
if MODE not in {"auto", "plot_only"} or set(METRICS) - set(UNITS) or not METRICS:
    raise ValueError("Check MODE and METRICS")
selection_inputs = [sorted(SEQUENCES), sorted(METHODS), sorted(METRICS), EXPERIMENTS, DEVICE, WORKERS, VOXEL_BINS, str(DATA)]
if set(METRICS).intersection(VOXEL_METRICS):
    selection_inputs += [PROTOCOL, VOXEL_BIN_MS]
selection = bench.digest(selection_inputs)[:16]
directory = REPORT / selection
if MODE == "plot_only":
    scores = pd.read_csv(directory / "per_sequence.csv")
else:
    scores = run_scores()
    with bench.atomic_file(directory / "per_sequence.csv") as f:
        scores.to_csv(f, index=False)
valid = scores.value.where(scores.status.eq("ok"))
summary = (scores.assign(valid=valid).groupby(["metric","method","stride"], as_index=False)
           .agg(mean=("valid","mean"), n_valid=("valid","count"), n_total=("sequence","size")))
for component in ("value_on", "value_off", "value_aware", "value_total"):
    if component in scores:
        means = (scores.assign(component=scores[component].where(scores.status.eq("ok")))
                 .groupby(["metric","method","stride"]).component.mean().rename(component))
        summary = summary.join(means, on=["metric","method","stride"])
if MODE == "auto":
    with bench.atomic_file(directory / "summary.csv") as f:
        summary.to_csv(f, index=False)
print(directory)
display(summary)
display(scores)

## Interactive comparison
Choose the mean or one sequence from the menu. Hover over a point for coverage. Click legend entries to toggle methods. Recorded GT is shown for descriptive metrics. Summary means include only finite scores with status ok; inspect coverage when comparing methods.

In [ ]:
for metric in METRICS:
    fig = go.Figure()
    choices = ["Mean", *SEQUENCES]
    labels = list(summary.loc[summary.metric == metric, "method"].unique())
    for i, choice in enumerate(choices):
        for method in labels:
            if i == 0:
                group = summary[(summary.metric == metric) & (summary.method == method)].sort_values("stride")
                y = group["mean"]
                details = group.n_valid.astype(str) + "/" + group.n_total.astype(str) + " valid"
            else:
                group = scores[(scores.sequence == choice) & (scores.metric == metric) & (scores.method == method)].sort_values("stride")
                y = group.value.where(group.status == "ok")
                details = group.status
            gt = method == "ground_truth"
            fig.add_scatter(x=group.stride, y=y, customdata=details, name="Recorded GT" if gt else method,
                            mode="lines+markers", visible=i == 0, connectgaps=False,
                            line=dict(dash="dash" if gt else "solid", color="black" if gt else None),
                            hovertemplate="Stride %{x}<br>%{y:.5g}<br>%{customdata}<extra>%{fullData.name}</extra>")
    buttons = [dict(label=c, method="update", args=[dict(visible=[j//len(labels) == i for j in range(len(fig.data))])])
               for i,c in enumerate(choices)]
    title = "Median ISI per sequence" if metric == "isi_distribution" else metric
    fig.update_layout(title=title, xaxis=dict(title="Frame stride", tickvals=sorted(EXPERIMENTS)),
                      yaxis_title=UNITS[metric], template="plotly_white",
                      updatemenus=[dict(buttons=buttons, x=0, y=1.2)])
    if metric == "event_count_ratio":
        fig.add_hline(y=1, line_dash="dot")
    fig.show()